# Validação dos pontos do Google Earth Pro

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import fiona
import contextily as ctx
from scipy.spatial import cKDTree

# ==============================================================================
# 1. CONFIGURAÇÕES E PARÂMETROS
# ==============================================================================
PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_ASC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
RAIO_IDW = 15      
RAIO_AMOSTRA = 15  
IDW_K, IDW_P = 3, 2
TARGET_VAR = 'dV_final'

# ==============================================================================
# 2. FUNÇÕES DE PROCESSAMENTO
# ==============================================================================
def melt_robust(df, val_name):
    meta = ['easting', 'northing', 'latitude', 'longitude', 'incidence_angle', 'track_angle', 'pid', 'p_id']
    present_meta = [c for c in meta if c in df.columns]
    date_cols = [c for c in df.columns if c not in meta]
    m = df.melt(id_vars=present_meta, value_vars=date_cols, var_name='date', value_name=val_name).dropna()
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    m[val_name] = pd.to_numeric(m[val_name], errors='coerce')
    return m.dropna(subset=['date', val_name])

def load_filter(path):
    df = pd.read_csv(path)
    return df[(df['northing'] >= norte_min-100) & (df['northing'] <= norte_max+100) & 
              (df['easting'] >= este_min-100) & (df['easting'] <= este_max+100)]

def interp_ps(df, dates):
    dfs = []
    t_x = dates.view(np.int64) 
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        xp, fp = g['date'].values.view(np.int64), g['disp'].values.astype(np.float64)
        res = pd.DataFrame({'easting': x, 'northing': y, 'date': dates, 'disp': np.interp(t_x, xp, fp),
                           'inc': g['incidence_angle'].iloc[0], 'lon': g['longitude'].iloc[0], 'lat': g['latitude'].iloc[0]})
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r, k, p):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        tree = cKDTree(src[['easting', 'northing']].values)
        dist, idx = tree.query(tgt[['easting', 'northing']].values, k=k, distance_upper_bound=r)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1 / (d_i[mask]**p)
            vals.append(np.sum(w * src.iloc[i_i[mask]]['disp']) / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[mask]]['inc']) / np.sum(w))
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna()

# ==============================================================================
# 3. EXECUÇÃO
# ==============================================================================
print("1/3 - Processando InSAR e Geometrias...")
asc_l = melt_robust(load_filter(PATH_ASC), 'disp')
desc_l = melt_robust(load_filter(PATH_DESC), 'disp')
common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')

asc_i = interp_ps(asc_l, common_dates)
desc_i = interp_ps(desc_l, common_dates)

# IDW e dV (O IDW é feito ANTES do dV)
manual_dv_df = idw_calc(desc_i, asc_i, r=RAIO_IDW, k=IDW_K, p=IDW_P)
manual_dv_df['dV_final'] = (manual_dv_df['disp_desc']*np.sin(np.deg2rad(manual_dv_df['inc'])) + 
                            manual_dv_df['disp']*np.sin(np.deg2rad(manual_dv_df['theta_desc']))) / \
                            np.sin(np.deg2rad(manual_dv_df['inc']) + np.deg2rad(manual_dv_df['theta_desc']))

gdf_manual = gpd.GeoDataFrame(manual_dv_df, geometry=gpd.points_from_xy(manual_dv_df['lon'], manual_dv_df['lat']), crs="EPSG:4326").to_crs(epsg=3763)

# Buffers KML
fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml = gpd.read_file(PATH_KML, driver='KML')
gdf_pts_geodesia = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_circulos = gdf_pts_geodesia.copy()
gdf_circulos.geometry = gdf_pts_geodesia.geometry.buffer(RAIO_AMOSTRA)

# ==============================================================================
# 4. FILTRAGEM APENAS PARA O MAPA (Pontos dentro dos raios)
# ==============================================================================
# Criamos GDFs específicos para as órbitas para ver onde estão os PS originais
gdf_asc_all = gpd.GeoDataFrame(asc_i.drop_duplicates(['easting','northing']), 
                               geometry=gpd.points_from_xy(asc_i.drop_duplicates(['easting','northing']).lon, 
                                                           asc_i.drop_duplicates(['easting','northing']).lat), crs="EPSG:4326").to_crs(epsg=3763)

gdf_desc_all = gpd.GeoDataFrame(desc_i.drop_duplicates(['easting','northing']), 
                                geometry=gpd.points_from_xy(desc_i.drop_duplicates(['easting','northing']).lon, 
                                                            desc_i.drop_duplicates(['easting','northing']).lat), crs="EPSG:4326").to_crs(epsg=3763)

# Selecionar apenas pontos que estão dentro de qualquer círculo
gdf_asc_in = gpd.sjoin(gdf_asc_all, gdf_circulos, how="inner", predicate="within")
gdf_desc_in = gpd.sjoin(gdf_desc_all, gdf_circulos, how="inner", predicate="within")

# ==============================================================================
# 5. VISUALIZAÇÃO FINAL
# ==============================================================================
fig, ax = plt.subplots(figsize=(12, 10))

# 1. Buffers de Amostragem (Vermelho Transparente)
gdf_circulos.to_crs(epsg=3857).plot(ax=ax, facecolor='red', alpha=0.2, edgecolor='black', linestyle='--', label='Área de Amostragem (15m)')

# 2. Órbita Ascendente DENTRO dos raios (Laranja)
if not gdf_asc_in.empty:
    gdf_asc_in.to_crs(epsg=3857).plot(ax=ax, color='#f39c12', markersize=25, label='PS Ascendente (no raio)', zorder=3)

# 3. Órbita Descendente DENTRO dos raios (Azul)
if not gdf_desc_in.empty:
    gdf_desc_in.to_crs(epsg=3857).plot(ax=ax, color='#3498db', markersize=25, label='PS Descendente (no raio)', zorder=3)

# 4. Pontos da Geodesia (Estrelas Amarelas)
gdf_pts_geodesia.to_crs(epsg=3857).plot(ax=ax, color='yellow', marker='*', markersize=180, edgecolor='black', label='Instrumento Geodesia', zorder=5)

# 5. Identificação (Labels)
for _, row in gdf_pts_geodesia.to_crs(epsg=3857).iterrows():
    ax.text(row.geometry.x, row.geometry.y + 12, row['Name'], color='white', fontweight='bold', 
            fontsize=9, ha='center', bbox=dict(facecolor='black', alpha=0.7, pad=2, edgecolor='none'))

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
plt.title(f"Diagnóstico InSAR Alqueva: PS capturados nos raios de {RAIO_AMOSTRA}m")
plt.legend(loc='lower right', frameon=True, facecolor='white')
ax.set_axis_off()
plt.tight_layout()
plt.show()

print(f"Sucesso: {len(gdf_asc_in)} pontos Asc e {len(gdf_desc_in)} pontos Desc identificados nos raios.")

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import contextily as ctx
import fiona
import matplotlib.dates as mdates
from shapely.geometry import box
from scipy.spatial import cKDTree
from statsmodels.tsa.seasonal import seasonal_decompose

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
TARGET_VAR = 'dV' 
PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_ASC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
PATH_ORTHO = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv" if TARGET_VAR == 'dV' else \
             "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
RAIO_IDW, RAIO_AMOSTRA = 50, 15
OFF_X, OFF_Y = 0, 0 

# ==============================================================================
# 2. FUNÇÕES DE PROCESSAMENTO (CORREÇÃO DE DATA)
# ==============================================================================
def melt_dynamic_robust(df, value_name):
    non_date_cols = ['easting', 'northing', 'latitude', 'longitude', 'incidence_angle', 
                    'track_angle', 'p_id', 'pid', 'altitude', 'velocity', 'coherence']
    present_metadata = [c for c in df.columns if c in non_date_cols]
    date_cols = [c for c in df.columns if c not in non_date_cols]
    m = df.melt(id_vars=present_metadata, value_vars=date_cols, var_name='date', value_name=value_name).dropna()
    
    # CORREÇÃO: Especificação do formato para evitar UserWarning
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    return m.dropna(subset=['date'])

def load_filter(path):
    df = pd.read_csv(path)
    return df[(df['northing'] >= norte_min-200) & (df['northing'] <= norte_max+200) & 
              (df['easting'] >= este_min-200) & (df['easting'] <= este_max+200)]

def interp_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(dates.astype(np.int64), g['date'].astype(np.int64), g['disp'])
        res = pd.DataFrame({'easting': x, 'northing': y, 'date': dates, 'disp': interp})
        res['lat'], res['lon'] = g['latitude'].iloc[0], g['longitude'].iloc[0]
        res['inc'] = g['incidence_angle'].iloc[0]
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r):
    if source.empty or target.empty: return pd.DataFrame()
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        tree = cKDTree(src[['easting', 'northing']].values)
        dist, idx = tree.query(tgt[['easting', 'northing']].values, k=5, distance_upper_bound=r)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1/(d_i[mask]**2)
            vals.append(np.sum(w*src.iloc[i_i[mask]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[mask]]['inc'])/np.sum(w))
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna()

print("1. Processando ficheiros...")
asc_raw, desc_raw, ortho_raw = load_filter(PATH_ASC), load_filter(PATH_DESC), load_filter(PATH_ORTHO)
asc_l, desc_l, ortho_l = melt_dynamic_robust(asc_raw, 'disp'), melt_dynamic_robust(desc_raw, 'disp'), melt_dynamic_robust(ortho_raw, 'disp_ortho')

common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')
asc_i, desc_i = interp_ps(asc_l, common_dates), interp_ps(desc_l, common_dates)

manual_dv_df = idw_calc(desc_i, asc_i, RAIO_IDW)
if not manual_dv_df.empty:
    manual_dv_df['dV_final'] = (manual_dv_df['disp_desc']*np.sin(np.deg2rad(manual_dv_df['inc'])) + 
                                manual_dv_df['disp']*np.sin(np.deg2rad(manual_dv_df['theta_desc']))) / \
                                np.sin(np.deg2rad(manual_dv_df['inc']) + np.deg2rad(manual_dv_df['theta_desc']))

# ==============================================================================
# 3. FILTRO KML E CAPTURA
# ==============================================================================
fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml = gpd.read_file(PATH_KML, driver='KML')
gdf_pts = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_pts.geometry = gdf_pts.geometry.translate(xoff=OFF_X, yoff=OFF_Y)
gdf_circs = gdf_pts.copy()
gdf_circs.geometry = gdf_pts.geometry.buffer(RAIO_AMOSTRA)

gdf_manual = gpd.GeoDataFrame(manual_dv_df, geometry=gpd.points_from_xy(manual_dv_df['lon'], manual_dv_df['lat']), crs="EPSG:4326").to_crs(epsg=3763) if not manual_dv_df.empty else gpd.GeoDataFrame()
gdf_ortho = gpd.GeoDataFrame(ortho_l, geometry=gpd.points_from_xy(ortho_l['longitude'] if 'longitude' in ortho_l else ortho_l['easting'], 
                                                                 ortho_l['latitude'] if 'latitude' in ortho_l else ortho_l['northing']), 
                             crs="EPSG:4326" if 'latitude' in ortho_l else "EPSG:3763").to_crs(epsg=3763)

final_series = {}
for nome in gdf_circs['Name'].unique():
    o_data = gpd.sjoin(gdf_ortho, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
    if not o_data.empty:
        print(f"-> {nome}: Usando Ortho")
        final_series[nome] = o_data.groupby('date')['disp_ortho'].mean()
    elif not gdf_manual.empty:
        m_data = gpd.sjoin(gdf_manual, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
        if not m_data.empty:
            print(f"-> {nome}: Usando Manual")
            final_series[nome] = m_data.groupby('date')['dV_final'].mean()

# ==============================================================================
# 4. FIGURA 1: MAPA DE DIAGNÓSTICO (CORREÇÃO DE LEGENDA E TAMANHO)
# ==============================================================================
print("2. Gerando Mapa de Diagnóstico...")
fig1, ax1 = plt.subplots(figsize=(12, 10))

# --- Pontos Órbita Ascendente (Redondos Vermelhos Pequenos) ---
u_asc = asc_i.drop_duplicates(['easting','northing'])
gdf_asc_map = gpd.GeoDataFrame(u_asc, geometry=gpd.points_from_xy(u_asc['lon'], u_asc['lat']), crs="EPSG:4326").to_crs(epsg=3857)
gdf_asc_map.plot(ax=ax1, color='red', marker='o', markersize=15, alpha=0.6)

# --- Pontos Órbita Descendente (Redondos Azuis Pequenos) ---
u_desc = desc_i.drop_duplicates(['easting','northing'])
gdf_desc_map = gpd.GeoDataFrame(u_desc, geometry=gpd.points_from_xy(u_desc['lon'], u_desc['lat']), crs="EPSG:4326").to_crs(epsg=3857)
gdf_desc_map.plot(ax=ax1, color='blue', marker='o', markersize=15, alpha=0.6)

# --- Círculos de Amostragem (Amarelo) ---
gdf_circs_map = gdf_circs.to_crs(epsg=3857)
gdf_circs_map.plot(ax=ax1, facecolor='none', edgecolor='yellow', lw=2.5, zorder=10)

# --- Rótulos (Labels) ---
for _, row in gdf_circs_map.iterrows():
    ax1.text(row.geometry.centroid.x, row.geometry.centroid.y + 12, row['Name'], 
             color='white', fontsize=10, fontweight='bold', ha='center',
             bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=2))

# --- CORREÇÃO DA LEGENDA (Proxy Artists) ---
handle_asc = mlines.Line2D([], [], color='red', marker='o', linestyle='None', markersize=6, label='Ascendente')
handle_desc = mlines.Line2D([], [], color='blue', marker='o', linestyle='None', markersize=6, label='Descendente')
handle_circ = mpatches.Patch(facecolor='none', edgecolor='yellow', linewidth=2, label='Círculos KML')

ax1.legend(handles=[handle_asc, handle_desc, handle_circ], loc='upper right', framealpha=0.9)

ctx.add_basemap(ax1, source=ctx.providers.Esri.WorldImagery)
lim_box = gpd.GeoDataFrame(geometry=[box(este_min, norte_min, este_max, norte_max)], crs="EPSG:3035").to_crs(epsg=3857)
minx, miny, maxx, maxy = lim_box.total_bounds
ax1.set_xlim(minx, maxx); ax1.set_ylim(miny, maxy)
ax1.set_axis_off()
plt.title("Diagnóstico Geográfico: Órbitas InSAR", fontsize=14)
plt.show()

# ==============================================================================
# 5/6. SÉRIES E STL (Inalterados, com fillna corrigido)
# ==============================================================================
if final_series:
    print("3. Gerando Grelha de Séries e STL...")
    nomes = list(final_series.keys())
    n_cells = len(nomes)
    cols = int(np.ceil(np.sqrt(n_cells)))
    rows = int(np.ceil(n_cells / cols))

    # Grelha de Séries
    fig2, axes2 = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), sharex=True, sharey=True)
    if n_cells == 1: axes2 = np.array([axes2])
    axes2 = axes2.flatten()
    all_vals_raw = pd.concat(final_series.values())
    ymin, ymax = all_vals_raw.min(), all_vals_raw.max()
    pad = (ymax - ymin) * 0.1

    for i in range(len(axes2)):
        ax = axes2[i]
        if i < n_cells:
            nome = nomes[i]
            # CORREÇÃO: ffill() e bfill() recomendados
            data = final_series[nome].interpolate().ffill().bfill()
            ax.plot(data.index, data.values, color='black', lw=1.2)
            ax.set_title(f"{nome}", fontsize=10, fontweight='bold', color='red')
            ax.set_ylim(ymin - pad, ymax + pad)
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
            if i % cols == 0: ax.set_ylabel(f'{TARGET_VAR} (mm)')
        else: ax.axis('off')
    plt.tight_layout(); plt.show()

    # STL Final
    decomps_storage = {}
    v_obs, v_trend, v_seas, v_resid = [], [], [], []
    for nome in nomes:
        series = final_series[nome].interpolate().ffill().bfill()
        try:
            res = seasonal_decompose(series, period=12, model='additive', extrapolate_trend='freq')
            decomps_storage[nome] = res
            v_obs.append(res.observed); v_trend.append(res.trend)
            v_seas.append(res.seasonal); v_resid.append(res.resid)
        except: decomps_storage[nome] = None

    def get_lims(v_list):
        all_v = pd.concat(v_list)
        margin = (all_v.max() - all_v.min()) * 0.1
        return (all_v.min() - margin, all_v.max() + margin)

    l_obs, l_trend, l_seas, l_resid = get_lims(v_obs), get_lims(v_trend), get_lims(v_seas), get_lims(v_resid)
    fig3, axes3 = plt.subplots(n_cells, 4, figsize=(16, 1.8 * n_cells), sharex=True, squeeze=False)

    for i in range(n_cells):
        nome = nomes[i]
        res = decomps_storage.get(nome)
        if res is None: continue
        axes3[i, 0].plot(res.observed.index, res.observed, color='black', lw=1); axes3[i, 0].set_ylim(l_obs)
        axes3[i, 0].set_ylabel(nome, fontweight='bold', color='red', fontsize=9)
        if i == 0: axes3[i, 0].set_title("Observed")
        axes3[i, 1].plot(res.trend.index, res.trend, color='blue', lw=1.5); axes3[i, 1].set_ylim(l_trend)
        if i == 0: axes3[i, 1].set_title("Trend")
        axes3[i, 2].plot(res.seasonal.index, res.seasonal, color='green', lw=1); axes3[i, 2].set_ylim(l_seas)
        if i == 0: axes3[i, 2].set_title("Seasonal")
        axes3[i, 3].scatter(res.resid.index, res.resid, color='gray', s=3, alpha=0.6); axes3[i, 3].set_ylim(l_resid)
        axes3[i, 3].axhline(0, c='k', ls='--', lw=0.5)
        if i == 0: axes3[i, 3].set_title("Residual")
        if i == n_cells - 1:
            for ax in axes3[i, :]: ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout(); plt.show()

# Comparação InSAR / Nivelamento LNEC

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import contextily as ctx
import fiona
import matplotlib.dates as mdates
from shapely.geometry import box
from scipy.spatial import cKDTree
from scipy import stats
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.seasonal import seasonal_decompose

# ==============================================================================
# 1. CONFIGURAÇÕES E CAMINHOS
# ==============================================================================
TARGET_VAR = 'dV' 
PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_NIVEL = 'data/nivelamento.xlsx'
PATH_ASC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
PATH_ORTHO = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
RAIO_IDW, RAIO_AMOSTRA = 50, 15
OFF_X, OFF_Y = 0, 0 

# ==============================================================================
# 2. FUNÇÕES DE PROCESSAMENTO
# ==============================================================================
def melt_robust(df, val_name):
    meta = ['easting', 'northing', 'latitude', 'longitude', 'incidence_angle', 'track_angle', 'pid', 'p_id']
    present_meta = [c for c in meta if c in df.columns]
    date_cols = [c for c in df.columns if c not in meta]
    m = df.melt(id_vars=present_meta, value_vars=date_cols, var_name='date', value_name=val_name).dropna()
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    return m.dropna(subset=['date'])

def load_filter(path):
    df = pd.read_csv(path)
    return df[(df['northing'] >= norte_min-200) & (df['northing'] <= norte_max+200) & 
              (df['easting'] >= este_min-200) & (df['easting'] <= este_max+200)]

def interp_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(dates.astype(np.int64), g['date'].astype(np.int64), g['disp'])
        res = pd.DataFrame({'easting': x, 'northing': y, 'date': dates, 'disp': interp,
                           'lat': g['latitude'].iloc[0], 'lon': g['longitude'].iloc[0], 'inc': g['incidence_angle'].iloc[0]})
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        tree = cKDTree(src[['easting', 'northing']].values)
        dist, idx = tree.query(tgt[['easting', 'northing']].values, k=5, distance_upper_bound=r)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1/(d_i[mask]**2)
            vals.append(np.sum(w*src.iloc[i_i[mask]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[mask]]['inc'])/np.sum(w))
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna()

print("1. Processando ficheiros InSAR...")
asc_raw, desc_raw, ortho_raw = load_filter(PATH_ASC), load_filter(PATH_DESC), load_filter(PATH_ORTHO)
asc_l, desc_l, ortho_l = melt_robust(asc_raw, 'disp'), melt_robust(desc_raw, 'disp'), melt_robust(ortho_raw, 'disp_ortho')

common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')
asc_i, desc_i = interp_ps(asc_l, common_dates), interp_ps(desc_l, common_dates)

manual_dv_df = idw_calc(desc_i, asc_i, RAIO_IDW)
if not manual_dv_df.empty:
    manual_dv_df['dV_final'] = (manual_dv_df['disp_desc']*np.sin(np.deg2rad(manual_dv_df['inc'])) + 
                                manual_dv_df['disp']*np.sin(np.deg2rad(manual_dv_df['theta_desc']))) / \
                                np.sin(np.deg2rad(manual_dv_df['inc']) + np.deg2rad(manual_dv_df['theta_desc']))

# ==============================================================================
# 3. PROCESSAMENTO GEODESIA
# ==============================================================================
print("2. Processando Geodesia...")
df_geo_raw = pd.read_excel(PATH_NIVEL)
df_geo_raw['data'] = pd.to_datetime(df_geo_raw['data'])
df_geo_raw['valor_corrigido'] = pd.to_numeric(df_geo_raw['deslocamento (m)'].astype(str).str.replace(',', '.'), errors='coerce')

# ==============================================================================
# 4. CAPTURA KML E UNIÃO
# ==============================================================================
fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml = gpd.read_file(PATH_KML, driver='KML')
gdf_pts = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_pts.geometry = gdf_pts.geometry.translate(xoff=OFF_X, yoff=OFF_Y)
gdf_circs = gdf_pts.copy()
gdf_circs.geometry = gdf_pts.geometry.buffer(RAIO_AMOSTRA)

gdf_manual = gpd.GeoDataFrame(manual_dv_df, geometry=gpd.points_from_xy(manual_dv_df['lon'], manual_dv_df['lat']), crs="EPSG:4326").to_crs(epsg=3763) if not manual_dv_df.empty else gpd.GeoDataFrame()
gdf_ortho = gpd.GeoDataFrame(ortho_l, geometry=gpd.points_from_xy(ortho_l['longitude'] if 'longitude' in ortho_l else ortho_l['easting'], 
                                                                 ortho_l['latitude'] if 'latitude' in ortho_l else ortho_l['northing']), 
                             crs="EPSG:4326" if 'latitude' in ortho_l else "EPSG:3763").to_crs(epsg=3763)

final_series = {}
for nome in gdf_circs['Name'].unique():
    o_data = gpd.sjoin(gdf_ortho, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
    if not o_data.empty:
        final_series[nome] = o_data.groupby('date')['disp_ortho'].mean()
    elif not gdf_manual.empty:
        m_data = gpd.sjoin(gdf_manual, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
        if not m_data.empty:
            final_series[nome] = m_data.groupby('date')['dV_final'].mean()

# ==============================================================================
# 5. FIGURA 1: MAPA DE DIAGNÓSTICO (Aumentado e com distinção de órbitas)
# ==============================================================================
fig1, ax1 = plt.subplots(figsize=(12, 10)) # Mapa maior

# Orbit Ascendente
u_asc = asc_i.drop_duplicates(['easting','northing'])
gdf_asc = gpd.GeoDataFrame(u_asc, geometry=gpd.points_from_xy(u_asc['lon'], u_asc['lat']), crs="EPSG:4326").to_crs(epsg=3857)
gdf_asc.plot(ax=ax1, color='red', marker='o', markersize=20, alpha=0.6, label='Ascendente')

# Orbit Descendente
u_desc = desc_i.drop_duplicates(['easting','northing'])
gdf_desc = gpd.GeoDataFrame(u_desc, geometry=gpd.points_from_xy(u_desc['lon'], u_desc['lat']), crs="EPSG:4326").to_crs(epsg=3857)
gdf_desc.plot(ax=ax1, color='blue', marker='o', markersize=20, alpha=0.6, label='Descendente')

# Círculos KML
gdf_circs_map = gdf_circs.to_crs(epsg=3857)
gdf_circs_map.plot(ax=ax1, facecolor='none', edgecolor='yellow', lw=2.5, zorder=10)

# Labels dos Pontos
for _, row in gdf_circs_map.iterrows():
    ax1.text(row.geometry.centroid.x, row.geometry.centroid.y + 15, row['Name'], 
             color='white', fontweight='bold', ha='center', fontsize=9,
             bbox=dict(facecolor='black', alpha=0.5, pad=1, edgecolor='none'))

ctx.add_basemap(ax1, source=ctx.providers.Esri.WorldImagery)
lim_box = gpd.GeoDataFrame(geometry=[box(este_min, norte_min, este_max, norte_max)], crs="EPSG:3035").to_crs(epsg=3857)
minx, miny, maxx, maxy = lim_box.total_bounds
ax1.set_xlim(minx, maxx); ax1.set_ylim(miny, maxy)
ax1.set_axis_off()
ax1.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.8)
plt.title("Localização dos Pontos e Diagnóstico de Órbitas InSAR", fontsize=14)
plt.show()

# ==============================================================================
# 6. ANÁLISE ESTATÍSTICA (Altura compacta, termo "Ponto")
# ==============================================================================
print("3. Gerando Análise Estatística...")
inst_comuns = [n for n in final_series.keys() if n in df_geo_raw['instrumento'].unique()]
stats_resumo = []

if inst_comuns:
    fig, axes = plt.subplots(len(inst_comuns), 1, figsize=(12, 3.5 * len(inst_comuns)), sharex=True)
    if len(inst_comuns) == 1: axes = [axes]
    
    for i, nome in enumerate(inst_comuns):
        ax = axes[i]
        s_insar = final_series[nome][final_series[nome].index.year >= 2019].sort_index()
        insar_zero = s_insar - s_insar.iloc[0]
        s_geo = df_geo_raw[(df_geo_raw['instrumento'] == nome) & (df_geo_raw['data'].dt.year >= 2019)].sort_values('data')
        geo_zero = s_geo['valor_corrigido'] - s_geo['valor_corrigido'].iloc[0]
        
        days_insar = (insar_zero.index - insar_zero.index[0]).days
        slope_i, intercept_i, r_val_i, _, _ = stats.linregress(days_insar, insar_zero.values)
        vel_insar = slope_i * 365.25
        days_geo = (s_geo['data'] - s_geo['data'].iloc[0]).dt.days
        slope_g, intercept_g, r_val_g, _, _ = stats.linregress(days_geo, geo_zero)
        vel_geo = slope_g * 365.25

        insar_interp = np.interp(days_geo, days_insar, insar_zero.values)
        rmse = np.sqrt(mean_squared_error(geo_zero, insar_interp))

        ax.plot(insar_zero.index, insar_zero.values, label=f'InSAR ({vel_insar:.2f} mm/ano)', color='tab:blue', alpha=0.4)
        ax.scatter(s_geo['data'], geo_zero, color='red', marker='D', s=50, label=f'Geodesia ({vel_geo:.2f} mm/ano)', zorder=5)
        ax.plot(insar_zero.index, intercept_i + slope_i * days_insar, color='darkblue', linestyle='--', lw=2)
        ax.plot(s_geo['data'], intercept_g + slope_g * days_geo, color='darkred', linestyle=':', lw=2)

        ax.text(0.02, 0.05, f"RMSE: {rmse:.2f} mm | $R^2$: {r_val_i**2:.2f}", transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'), fontsize=9)
        ax.set_title(f"Validação Ponto {nome}", fontsize=11, fontweight='bold')
        ax.set_ylabel("Variação (mm)")
        ax.legend(loc='upper left', fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)

        stats_resumo.append({'Ponto': nome, 'Vel_InSAR (mm/ano)': vel_insar, 'Vel_Geo (mm/ano)': vel_geo, 'RMSE (mm)': rmse})

    plt.tight_layout()
    plt.show()
    print("\n--- TABELA RESUMO ---")
    print(pd.DataFrame(stats_resumo).to_string(index=False))

## com definição dos parâmetros IDW

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import contextily as ctx
import fiona
import matplotlib.dates as mdates
from shapely.geometry import box
from scipy.spatial import cKDTree
from scipy import stats
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.seasonal import seasonal_decompose

# ==============================================================================
# 1. CONFIGURAÇÕES E CAMINHOS
# ==============================================================================
TARGET_VAR = 'dV' 
PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_NIVEL = 'data/nivelamento.xlsx'
PATH_ASC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
PATH_ORTHO = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# Coordenadas de corte
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

# --- PARÂMETROS ESPACIAIS E IDW ---
RAIO_IDW = 15      # Distância máxima (m) para procurar vizinhos durante a interpolação IDW
RAIO_AMOSTRA = 15  # Tamanho do círculo (m) em torno dos pontos KML para extrair a média InSAR
IDW_K = 3          # Número de vizinhos mais próximos a usar no cálculo IDW
IDW_P = 2          # Coeficiente de potência (2 = inverso do quadrado da distância)
OFF_X, OFF_Y = 0, 0 

# ==============================================================================
# 2. FUNÇÕES DE PROCESSAMENTO
# ==============================================================================
def melt_robust(df, val_name):
    meta = ['easting', 'northing', 'latitude', 'longitude', 'incidence_angle', 'track_angle', 'pid', 'p_id']
    present_meta = [c for c in meta if c in df.columns]
    date_cols = [c for c in df.columns if c not in meta]
    m = df.melt(id_vars=present_meta, value_vars=date_cols, var_name='date', value_name=val_name).dropna()
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    return m.dropna(subset=['date'])

def load_filter(path):
    df = pd.read_csv(path)
    return df[(df['northing'] >= norte_min-200) & (df['northing'] <= norte_max+200) & 
              (df['easting'] >= este_min-200) & (df['easting'] <= este_max+200)]

def interp_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(dates.astype(np.int64), g['date'].astype(np.int64), g['disp'])
        res = pd.DataFrame({'easting': x, 'northing': y, 'date': dates, 'disp': interp,
                           'lat': g['latitude'].iloc[0], 'lon': g['longitude'].iloc[0], 'inc': g['incidence_angle'].iloc[0]})
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r, k=5, p=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        tree = cKDTree(src[['easting', 'northing']].values)
        
        # Procura os k vizinhos dentro do raio r
        dist, idx = tree.query(tgt[['easting', 'northing']].values, k=k, distance_upper_bound=r)
        
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): 
                vals.append(np.nan); thetas.append(np.nan); continue
            
            # Cálculo dos pesos baseado na potência p
            w = 1 / (d_i[mask]**p)
            
            vals.append(np.sum(w * src.iloc[i_i[mask]]['disp']) / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[mask]]['inc']) / np.sum(w))
            
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna()

# --- Execução do Processamento ---
print("1. Processando ficheiros InSAR...")
asc_raw, desc_raw, ortho_raw = load_filter(PATH_ASC), load_filter(PATH_DESC), load_filter(PATH_ORTHO)
asc_l, desc_l, ortho_l = melt_robust(asc_raw, 'disp'), melt_robust(desc_raw, 'disp'), melt_robust(ortho_raw, 'disp_ortho')

common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')
asc_i, desc_i = interp_ps(asc_l, common_dates), interp_ps(desc_l, common_dates)

print(f"   Aplicando IDW (r={RAIO_IDW}m, k={IDW_K}, p={IDW_P})...")
manual_dv_df = idw_calc(desc_i, asc_i, RAIO_IDW, k=IDW_K, p=IDW_P)

if not manual_dv_df.empty:
    manual_dv_df['dV_final'] = (manual_dv_df['disp_desc']*np.sin(np.deg2rad(manual_dv_df['inc'])) + 
                                manual_dv_df['disp']*np.sin(np.deg2rad(manual_dv_df['theta_desc']))) / \
                                np.sin(np.deg2rad(manual_dv_df['inc']) + np.deg2rad(manual_dv_df['theta_desc']))

# ==============================================================================
# 3. PROCESSAMENTO GEODESIA
# ==============================================================================
print("2. Processando Geodesia...")
df_geo_raw = pd.read_excel(PATH_NIVEL)
df_geo_raw['data'] = pd.to_datetime(df_geo_raw['data'])
df_geo_raw['valor_corrigido'] = pd.to_numeric(df_geo_raw['deslocamento (m)'].astype(str).str.replace(',', '.'), errors='coerce')

# ==============================================================================
# 4. CAPTURA KML E UNIÃO ESPACIAL
# ==============================================================================
fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml = gpd.read_file(PATH_KML, driver='KML')
gdf_pts = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_pts.geometry = gdf_pts.geometry.translate(xoff=OFF_X, yoff=OFF_Y)

# Criação dos buffers de amostragem
gdf_circs = gdf_pts.copy()
gdf_circs.geometry = gdf_pts.geometry.buffer(RAIO_AMOSTRA)

gdf_manual = gpd.GeoDataFrame(manual_dv_df, geometry=gpd.points_from_xy(manual_dv_df['lon'], manual_dv_df['lat']), crs="EPSG:4326").to_crs(epsg=3763) if not manual_dv_df.empty else gpd.GeoDataFrame()
gdf_ortho = gpd.GeoDataFrame(ortho_l, geometry=gpd.points_from_xy(ortho_l['longitude'] if 'longitude' in ortho_l else ortho_l['easting'], 
                                                                 ortho_l['latitude'] if 'latitude' in ortho_l else ortho_l['northing']), 
                               crs="EPSG:4326" if 'latitude' in ortho_l else "EPSG:3763").to_crs(epsg=3763)

final_series = {}
for nome in gdf_circs['Name'].unique():
    o_data = gpd.sjoin(gdf_ortho, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
    if not o_data.empty:
        final_series[nome] = o_data.groupby('date')['disp_ortho'].mean()
    elif not gdf_manual.empty:
        m_data = gpd.sjoin(gdf_manual, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
        if not m_data.empty:
            final_series[nome] = m_data.groupby('date')['dV_final'].mean()

# ==============================================================================
# 5. FIGURA 1: MAPA DE DIAGNÓSTICO
# ==============================================================================
fig1, ax1 = plt.subplots(figsize=(12, 10))
u_asc = asc_i.drop_duplicates(['easting','northing'])
gdf_asc = gpd.GeoDataFrame(u_asc, geometry=gpd.points_from_xy(u_asc['lon'], u_asc['lat']), crs="EPSG:4326").to_crs(epsg=3857)
gdf_asc.plot(ax=ax1, color='red', marker='o', markersize=20, alpha=0.6, label='Ascendente')

u_desc = desc_i.drop_duplicates(['easting','northing'])
gdf_desc = gpd.GeoDataFrame(u_desc, geometry=gpd.points_from_xy(u_desc['lon'], u_desc['lat']), crs="EPSG:4326").to_crs(epsg=3857)
gdf_desc.plot(ax=ax1, color='blue', marker='o', markersize=20, alpha=0.6, label='Descendente')

gdf_circs_map = gdf_circs.to_crs(epsg=3857)
gdf_circs_map.plot(ax=ax1, facecolor='none', edgecolor='yellow', lw=2.5, zorder=10)

for _, row in gdf_circs_map.iterrows():
    ax1.text(row.geometry.centroid.x, row.geometry.centroid.y + 15, row['Name'], 
             color='white', fontweight='bold', ha='center', fontsize=9,
             bbox=dict(facecolor='black', alpha=0.5, pad=1, edgecolor='none'))

ctx.add_basemap(ax1, source=ctx.providers.Esri.WorldImagery)
lim_box = gpd.GeoDataFrame(geometry=[box(este_min, norte_min, este_max, norte_max)], crs="EPSG:3035").to_crs(epsg=3857)
minx, miny, maxx, maxy = lim_box.total_bounds
ax1.set_xlim(minx, maxx); ax1.set_ylim(miny, maxy)
ax1.set_axis_off()
ax1.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.8)
plt.title("Localização dos Pontos e Diagnóstico de Órbitas InSAR", fontsize=14)
plt.show()

# ==============================================================================
# 6. ANÁLISE ESTATÍSTICA E VALIDAÇÃO
# ==============================================================================
print("3. Gerando Análise Estatística...")
inst_comuns = [n for n in final_series.keys() if n in df_geo_raw['instrumento'].unique()]
stats_resumo = []

if inst_comuns:
    fig, axes = plt.subplots(len(inst_comuns), 1, figsize=(12, 3.5 * len(inst_comuns)), sharex=True)
    if len(inst_comuns) == 1: axes = [axes]
    
    for i, nome in enumerate(inst_comuns):
        ax = axes[i]
        s_insar = final_series[nome][final_series[nome].index.year >= 2019].sort_index()
        insar_zero = s_insar - s_insar.iloc[0]
        s_geo = df_geo_raw[(df_geo_raw['instrumento'] == nome) & (df_geo_raw['data'].dt.year >= 2019)].sort_values('data')
        geo_zero = s_geo['valor_corrigido'] - s_geo['valor_corrigido'].iloc[0]
        
        days_insar = (insar_zero.index - insar_zero.index[0]).days
        slope_i, intercept_i, r_val_i, _, _ = stats.linregress(days_insar, insar_zero.values)
        vel_insar = slope_i * 365.25
        
        days_geo = (s_geo['data'] - s_geo['data'].iloc[0]).dt.days
        slope_g, intercept_g, r_val_g, _, _ = stats.linregress(days_geo, geo_zero)
        vel_geo = slope_g * 365.25

        insar_interp = np.interp(days_geo, days_insar, insar_zero.values)
        rmse = np.sqrt(mean_squared_error(geo_zero, insar_interp))

        ax.plot(insar_zero.index, insar_zero.values, label=f'InSAR ({vel_insar:.2f} mm/ano)', color='tab:blue', alpha=0.4)
        ax.scatter(s_geo['data'], geo_zero, color='red', marker='D', s=50, label=f'Geodesia ({vel_geo:.2f} mm/ano)', zorder=5)
        ax.plot(insar_zero.index, intercept_i + slope_i * days_insar, color='darkblue', linestyle='--', lw=2)
        ax.plot(s_geo['data'], intercept_g + slope_g * days_geo, color='darkred', linestyle=':', lw=2)

        ax.text(0.02, 0.05, f"RMSE: {rmse:.2f} mm | $R^2$: {r_val_i**2:.2f}", transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'), fontsize=9)
        ax.set_title(f"Validação Ponto {nome}", fontsize=11, fontweight='bold')
        ax.set_ylabel("Variação (mm)")
        ax.legend(loc='upper left', fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)

        stats_resumo.append({'Ponto': nome, 'Vel_InSAR (mm/ano)': vel_insar, 'Vel_Geo (mm/ano)': vel_geo, 'RMSE (mm)': rmse})

    plt.tight_layout()
    plt.show()
    print("\n--- TABELA RESUMO ---")
    print(pd.DataFrame(stats_resumo).to_string(index=False))

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import fiona
import contextily as ctx
from scipy.spatial import cKDTree
from scipy import stats
from sklearn.metrics import mean_squared_error

# ==============================================================================
# 1. CONFIGURAÇÕES, CAMINHOS E PARÂMETROS
# ==============================================================================
PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_NIVEL = 'data/nivelamento.xlsx'
PATH_ASC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# ROI: Alqueva (EPSG:3763)
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
RAIO_AMOSTRA = 15 

# Cenários para análise de sensibilidade
CENARIOS = [
    {'id': 'Largo (50m)', 'r': 50, 'k': 5, 'p': 2},
    {'id': 'Médio (30m)', 'r': 30, 'k': 4, 'p': 2},
    {'id': 'Apertado (15m)', 'r': 15, 'k': 3, 'p': 2}
]

# ==============================================================================
# 2. FUNÇÕES DE PROCESSAMENTO
# ==============================================================================
def melt_robust(df, val_name):
    meta = ['easting', 'northing', 'latitude', 'longitude', 'incidence_angle', 'track_angle', 'pid', 'p_id']
    present_meta = [c for c in meta if c in df.columns]
    date_cols = [c for c in df.columns if c not in present_meta]
    m = df.melt(id_vars=present_meta, value_vars=date_cols, var_name='date', value_name=val_name).dropna()
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    m[val_name] = pd.to_numeric(m[val_name], errors='coerce')
    return m.dropna(subset=['date', val_name])

def load_filter(path):
    df = pd.read_csv(path)
    df['easting'] = pd.to_numeric(df['easting'], errors='coerce')
    df['northing'] = pd.to_numeric(df['northing'], errors='coerce')
    return df[(df['northing'] >= norte_min-100) & (df['northing'] <= norte_max+100) & 
              (df['easting'] >= este_min-100) & (df['easting'] <= este_max+100)].dropna(subset=['easting', 'northing'])

def interp_ps(df, dates):
    dfs = []
    target_x = dates.view(np.int64) 
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        xp, fp = g['date'].values.view(np.int64), g['disp'].values.astype(np.float64)
        res = pd.DataFrame({'easting': x, 'northing': y, 'date': dates, 'disp': np.interp(target_x, xp, fp),
                           'inc': g['incidence_angle'].iloc[0], 'lon': g['longitude'].iloc[0], 'lat': g['latitude'].iloc[0]})
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r, k, p):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(src[['easting', 'northing']].values)
        dist, idx = tree.query(tgt[['easting', 'northing']].values, k=k, distance_upper_bound=r)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1 / (d_i[mask]**p)
            vals.append(np.sum(w * src.iloc[i_i[mask]]['disp']) / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[mask]]['inc']) / np.sum(w))
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna()

# ==============================================================================
# 3. CARREGAMENTO E INTERPOLAÇÃO TEMPORAL
# ==============================================================================
print("Iniciando processamento base...")
asc_l = melt_robust(load_filter(PATH_ASC), 'disp')
desc_l = melt_robust(load_filter(PATH_DESC), 'disp')
common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')

asc_i = interp_ps(asc_l, common_dates)
desc_i = interp_ps(desc_l, common_dates)

# Geodesia e KML
df_geo = pd.read_excel(PATH_NIVEL)
df_geo['data'] = pd.to_datetime(df_geo['data'])
df_geo['val'] = pd.to_numeric(df_geo['deslocamento (m)'].astype(str).str.replace(',', '.'), errors='coerce')

fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml = gpd.read_file(PATH_KML, driver='KML')
gdf_pts_geo = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_circs = gdf_pts_geo.copy()
gdf_circs.geometry = gdf_pts_geo.geometry.buffer(RAIO_AMOSTRA)

# ==============================================================================
# 4. LOOP DE SENSIBILIDADE E ANÁLISE FINA
# ==============================================================================
res_stats = []
melhor_gdf_manual = None

for c in CENARIOS:
    print(f"Processando {c['id']}...")
    # IDW entre órbitas (Desc -> Asc)
    df_dv = idw_calc(desc_i, asc_i, r=c['r'], k=c['k'], p=c['p'])
    
    # Cálculo Vertical Final (Decomposição)
    df_dv['dV'] = (df_dv['disp_desc']*np.sin(np.deg2rad(df_dv['inc'])) + 
                   df_dv['disp']*np.sin(np.deg2rad(df_dv['theta_desc']))) / \
                   np.sin(np.deg2rad(df_dv['inc']) + np.deg2rad(df_dv['theta_desc']))
    
    gdf_manual = gpd.GeoDataFrame(df_dv, geometry=gpd.points_from_xy(df_dv.lon, df_dv.lat), crs="EPSG:4326").to_crs(epsg=3763)
    if c['id'] == 'Apertado (15m)': melhor_gdf_manual = gdf_manual.copy()

    # Validação Estatística
    for nome in gdf_circs['Name'].unique():
        joined = gpd.sjoin(gdf_manual, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
        if not joined.empty:
            s_insar = joined.groupby('date')['dV'].mean().sort_index()
            insar_z = (s_insar - s_insar.iloc[0]).values
            
            s_geo = df_geo[df_geo['instrumento'] == nome].sort_values('data')
            if len(s_geo) < 2: continue
            geo_z = (s_geo['val'] - s_geo['val'].iloc[0]).values
            
            # Alinhamento para métricas
            d_insar, d_geo = (s_insar.index - s_insar.index[0]).days, (s_geo['data'] - s_geo['data'].iloc[0]).dt.days
            insar_interp = np.interp(d_geo, d_insar, insar_z)
            
            # Métricas
            rmse = np.sqrt(mean_squared_error(geo_z, insar_interp))
            slope_i, _, r_v, _, _ = stats.linregress(d_geo, insar_interp)
            slope_g, _, _, _, _ = stats.linregress(d_geo, geo_z)
            
            res_stats.append({
                'Ponto': nome, 'Cenário': c['id'], 'RMSE': rmse, 'R2': r_v**2,
                'Vel_InSAR': slope_i*365.25, 'Vel_Geo': slope_g*365.25
            })

df_metrics = pd.DataFrame(res_stats)

# ==============================================================================
# 5. VISUALIZAÇÃO DE DIAGNÓSTICO (PS NO RAIO) E MÉTRICAS
# ==============================================================================
# Gráfico 1: Comparação de RMSE por Cenário
pivot_rmse = df_metrics.pivot(index='Ponto', columns='Cenário', values='RMSE')
pivot_rmse.plot(kind='bar', figsize=(12, 5), color=['#95a5a6', '#3498db', '#e74c3c'], width=0.8)
plt.title("Validação Sensibilidade: RMSE (mm)")
plt.axhline(y=2.0, color='green', linestyle='--', alpha=0.5)
plt.show()

# Gráfico 2: Mapa de Diagnóstico (Apenas pontos dentro dos raios no cenário Apertado)
fig, ax = plt.subplots(figsize=(10, 10))
gdf_circs.to_crs(epsg=3857).plot(ax=ax, facecolor='red', alpha=0.15, edgecolor='black', linestyle='--')

# Filtrar para visualização das órbitas originais nos raios
gdf_asc_all = gpd.GeoDataFrame(asc_i.drop_duplicates(['easting','northing']), 
                               geometry=gpd.points_from_xy(asc_i.drop_duplicates(['easting','northing']).lon, 
                               asc_i.drop_duplicates(['easting','northing']).lat), crs="EPSG:4326").to_crs(epsg=3857)
gdf_desc_all = gpd.GeoDataFrame(desc_i.drop_duplicates(['easting','northing']), 
                                geometry=gpd.points_from_xy(desc_i.drop_duplicates(['easting','northing']).lon, 
                                desc_i.drop_duplicates(['easting','northing']).lat), crs="EPSG:4326").to_crs(epsg=3857)

gpd.sjoin(gdf_asc_all, gdf_circs.to_crs(epsg=3857), how="inner").plot(ax=ax, color='#f39c12', markersize=30, label='PS Ascendente')
gpd.sjoin(gdf_desc_all, gdf_circs.to_crs(epsg=3857), how="inner").plot(ax=ax, color='#3498db', markersize=30, label='PS Descendente')
gdf_pts_geo.to_crs(epsg=3857).plot(ax=ax, color='yellow', marker='*', markersize=200, edgecolor='black', label='Instrumento Geodésico')

for _, row in gdf_pts_geo.to_crs(epsg=3857).iterrows():
    ax.text(row.geometry.x, row.geometry.y + 12, row['Name'], color='white', fontweight='bold', fontsize=9, ha='center', bbox=dict(facecolor='black', alpha=0.7, pad=1))

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
plt.title("Mapa de Diagnóstico: Distribuição de Órbitas nos Instrumentos")
plt.legend(loc='lower right')
ax.set_axis_off()
plt.show()

print("\n--- RESUMO DAS MELHORES MÉTRICAS POR PONTO ---")
print(df_metrics.sort_values(['Ponto', 'R2'], ascending=[True, False]).groupby('Ponto').head(1).to_string(index=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from scipy import stats

# ==============================================================================
# 6. ANÁLISE HISTÓRICA COM TENDÊNCIA LINEAR (SINC. 2019)
# ==============================================================================
print("Gerando Comparação Histórica com Tendência Linear...")

inst_comuns = [n for n in final_series.keys() if n in df_geo_raw['instrumento'].unique()]

if inst_comuns:
    # Altura compacta por ponto (3.5)
    fig, axes = plt.subplots(len(inst_comuns), 1, figsize=(14, 3.8 * len(inst_comuns)), sharex=False)
    if len(inst_comuns) == 1: axes = [axes]
    
    for i, nome in enumerate(inst_comuns):
        ax = axes[i]
        
        # --- 1. PROCESSAR InSAR (2019-2023) ---
        s_insar = final_series[nome][final_series[nome].index.year >= 2019].sort_index()
        insar_zero = s_insar - s_insar.iloc[0]
        
        # --- 2. PROCESSAR GEODESIA HISTÓRICA (2004-2023) ---
        s_geo_full = df_geo_raw[df_geo_raw['instrumento'] == nome].sort_values('data')
        
        # Encontrar referência de 2019
        s_geo_2019 = s_geo_full[s_geo_full['data'].dt.year >= 2019]
        
        if not s_geo_2019.empty and not s_insar.empty:
            ref_2019 = s_geo_2019['valor_corrigido'].iloc[0]
            geo_hist_zero = s_geo_full['valor_corrigido'] - ref_2019
            
            # --- 3. CÁLCULO DE TENDÊNCIAS LINEARES (2019-2023) ---
            # Dias para regressão
            days_insar = (insar_zero.index - insar_zero.index[0]).days
            slope_i, intercept_i, r_val_i, _, _ = stats.linregress(days_insar, insar_zero.values)
            
            days_geo_2019 = (s_geo_2019['data'] - s_geo_2019['data'].iloc[0]).dt.days
            vals_geo_2019 = s_geo_2019['valor_corrigido'] - ref_2019
            slope_g, intercept_g, r_val_g, _, _ = stats.linregress(days_geo_2019, vals_geo_2019)

            # --- 4. PLOTAGEM ---
            # Geodesia Pré-2019
            mask_pre = s_geo_full['data'].dt.year < 2019
            ax.plot(s_geo_full['data'][mask_pre], geo_hist_zero[mask_pre], 
                    color='gray', linestyle='--', alpha=0.4, label='Histórico Geodesia (2004-2018)')
            
            # Geodesia Pós-2019
            mask_pos = s_geo_full['data'].dt.year >= 2019
            ax.scatter(s_geo_full['data'][mask_pos], geo_hist_zero[mask_pos], 
                       color='red', marker='D', s=35, label='Geodesia (2019-2023)', zorder=5)
            
            # InSAR
            ax.plot(insar_zero.index, insar_zero.values, color='tab:blue', lw=1.2, label='InSAR (2019-2023)')

            # Linhas de Tendência Linear (Apenas para o período comum)
            ax.plot(insar_zero.index, intercept_i + slope_i * days_insar, 
                    color='darkblue', lw=2, label=f'Trend InSAR ({slope_i*365.25:.2f} mm/ano)')
            ax.plot(s_geo_2019['data'], intercept_g + slope_g * days_geo_2019, 
                    color='darkred', ls=':', lw=2, label=f'Trend Geodesia ({slope_g*365.25:.2f} mm/ano)')

            # --- FORMATATAÇÃO ---
            ax.axhline(0, color='black', lw=1)
            ax.axvline(pd.Timestamp('2019-01-01'), color='green', lw=1.5, ls='--', alpha=0.6)
            
            ax.set_title(f"Ponto {nome}: Comparação de Velocidades (Zero em 2019)", fontsize=11, fontweight='bold')
            ax.set_ylabel("Variação (mm)")
            ax.grid(True, alpha=0.2)
            ax.legend(loc='upper left', fontsize=8, ncol=2)
            
            # Ajustar eixo X para ver todo o histórico
            ax.xaxis.set_major_locator(mdates.YearLocator(2))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import fiona
import contextily as ctx
from scipy.spatial import cKDTree
from scipy import stats
from sklearn.metrics import mean_squared_error

# ==============================================================================
# 1. CONFIGURAÇÕES E PARÂMETROS
# ==============================================================================
PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_NIVEL = 'data/nivelamento.xlsx'
PATH_ASC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# Parâmetros Espaciais
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
RAIO_IDW, RAIO_AMOSTRA = 15, 15
IDW_K, IDW_P = 3, 2
TARGET_VAR = 'dV_final'

# ==============================================================================
# 2. FUNÇÕES DE PROCESSAMENTO
# ==============================================================================
def melt_robust(df, val_name):
    meta = ['easting', 'northing', 'latitude', 'longitude', 'incidence_angle', 'track_angle', 'pid', 'p_id']
    present_meta = [c for c in meta if c in df.columns]
    date_cols = [c for c in df.columns if c not in present_meta]
    m = df.melt(id_vars=present_meta, value_vars=date_cols, var_name='date', value_name=val_name).dropna()
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    m[val_name] = pd.to_numeric(m[val_name], errors='coerce')
    return m.dropna(subset=['date', val_name])

def load_filter(path):
    df = pd.read_csv(path)
    df['easting'] = pd.to_numeric(df['easting'], errors='coerce')
    df['northing'] = pd.to_numeric(df['northing'], errors='coerce')
    return df[(df['northing'] >= norte_min-100) & (df['northing'] <= norte_max+100) & 
              (df['easting'] >= este_min-100) & (df['easting'] <= este_max+100)].dropna(subset=['easting', 'northing'])

def interp_ps(df, dates):
    dfs = []
    t_x = dates.view(np.int64) 
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        xp, fp = g['date'].values.view(np.int64), g['disp'].values.astype(np.float64)
        res = pd.DataFrame({'easting': x, 'northing': y, 'date': dates, 'disp': np.interp(t_x, xp, fp),
                           'inc': g['incidence_angle'].iloc[0], 'lon': g['longitude'].iloc[0], 'lat': g['latitude'].iloc[0]})
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r, k, p):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(src[['easting', 'northing']].values)
        dist, idx = tree.query(tgt[['easting', 'northing']].values, k=k, distance_upper_bound=r)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1 / (d_i[mask]**p)
            vals.append(np.sum(w * src.iloc[i_i[mask]]['disp']) / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[mask]]['inc']) / np.sum(w))
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna()

# ==============================================================================
# 3. PROCESSAMENTO InSAR
# ==============================================================================
print("1/5 - Processando Séries InSAR e IDW...")
asc_l = melt_robust(load_filter(PATH_ASC), 'disp')
desc_l = melt_robust(load_filter(PATH_DESC), 'disp')
common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')

asc_i = interp_ps(asc_l, common_dates)
desc_i = interp_ps(desc_l, common_dates)

manual_dv_df = idw_calc(desc_i, asc_i, r=RAIO_IDW, k=IDW_K, p=IDW_P)
manual_dv_df['dV_final'] = (manual_dv_df['disp_desc']*np.sin(np.deg2rad(manual_dv_df['inc'])) + 
                            manual_dv_df['disp']*np.sin(np.deg2rad(manual_dv_df['theta_desc']))) / \
                            np.sin(np.deg2rad(manual_dv_df['inc']) + np.deg2rad(manual_dv_df['theta_desc']))

gdf_manual = gpd.GeoDataFrame(manual_dv_df, geometry=gpd.points_from_xy(manual_dv_df.lon, manual_dv_df.lat), crs="EPSG:4326").to_crs(epsg=3763)

# ==============================================================================
# 4. CARREGAMENTO GEODESIA E KML
# ==============================================================================
print("2/5 - Carregando Geodesia e buffers KML...")
df_geo_raw = pd.read_excel(PATH_NIVEL)
df_geo_raw['data'] = pd.to_datetime(df_geo_raw['data'])
df_geo_raw['valor_corrigido'] = pd.to_numeric(df_geo_raw['deslocamento (m)'].astype(str).str.replace(',', '.'), errors='coerce')

fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml = gpd.read_file(PATH_KML, driver='KML')
gdf_pts = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_circs = gdf_pts.copy()
gdf_circs.geometry = gdf_pts.geometry.buffer(RAIO_AMOSTRA)

# Amostragem Espacial (Criar dicionário final_series)
final_series = {}
for nome in gdf_circs['Name'].unique():
    m_data = gpd.sjoin(gdf_manual, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
    if not m_data.empty:
        final_series[nome] = m_data.groupby('date')['dV_final'].mean()

# ==============================================================================
# 5. ANÁLISE HISTÓRICA E GRÁFICOS (SINC. 2019)
# ==============================================================================
print("3/5 - Gerando Comparação Histórica e Tendências...")
inst_comuns = [n for n in final_series.keys() if n in df_geo_raw['instrumento'].unique()]
stats_resumo = []

if inst_comuns:
    fig, axes = plt.subplots(len(inst_comuns), 1, figsize=(14, 4 * len(inst_comuns)))
    if len(inst_comuns) == 1: axes = [axes]
    
    for i, nome in enumerate(inst_comuns):
        ax = axes[i]
        
        # --- 1. PROCESSAR InSAR (2019-2023) ---
        s_insar = final_series[nome][final_series[nome].index.year >= 2019].sort_index()
        insar_zero = s_insar - s_insar.iloc[0]
        
        # --- 2. PROCESSAR GEODESIA HISTÓRICA ---
        s_geo_full = df_geo_raw[df_geo_raw['instrumento'] == nome].sort_values('data')
        s_geo_2019 = s_geo_full[s_geo_full['data'].dt.year >= 2019]
        
        if not s_geo_2019.empty and not s_insar.empty:
            ref_2019 = s_geo_2019['valor_corrigido'].iloc[0]
            geo_hist_zero = s_geo_full['valor_corrigido'] - ref_2019
            
            # --- 3. TENDÊNCIAS (2019-2023) ---
            days_insar = (insar_zero.index - insar_zero.index[0]).days
            slope_i, intercept_i, r_val_i, _, _ = stats.linregress(days_insar, insar_zero.values)
            
            days_geo_2019 = (s_geo_2019['data'] - s_geo_2019['data'].iloc[0]).dt.days
            vals_geo_2019 = s_geo_2019['valor_corrigido'] - ref_2019
            slope_g, intercept_g, r_val_g, _, _ = stats.linregress(days_geo_2019, vals_geo_2019)

            # Cálculo de RMSE (Alinhamento temporal)
            insar_interp = np.interp(days_geo_2019, days_insar, insar_zero.values)
            rmse = np.sqrt(mean_squared_error(vals_geo_2019, insar_interp))

            # --- 4. PLOTAGEM ---
            mask_pre = s_geo_full['data'].dt.year < 2019
            ax.plot(s_geo_full['data'][mask_pre], geo_hist_zero[mask_pre], color='gray', ls='--', alpha=0.4, label='Histórico (2004-2018)')
            
            mask_pos = s_geo_full['data'].dt.year >= 2019
            ax.scatter(s_geo_full['data'][mask_pos], geo_hist_zero[mask_pos], color='red', marker='D', s=35, label='Geo 2019-2023', zorder=5)
            ax.plot(insar_zero.index, insar_zero.values, color='tab:blue', lw=1.2, label='InSAR 2019-2023')

            # Linhas de Tendência
            ax.plot(insar_zero.index, intercept_i + slope_i * days_insar, color='darkblue', lw=2, label=f'Trend InSAR ({slope_i*365.25:.2f} mm/a)')
            ax.plot(s_geo_2019['data'], intercept_g + slope_g * days_geo_2019, color='darkred', ls=':', lw=2, label=f'Trend Geo ({slope_g*365.25:.2f} mm/a)')

            # Formatação
            ax.axhline(0, color='black', lw=1)
            ax.axvline(pd.Timestamp('2019-01-01'), color='green', lw=1.5, ls='--', alpha=0.6)
            ax.set_title(f"Ponto {nome}: Análise de Tendência", fontweight='bold')
            ax.set_ylabel("Variação (mm)")
            ax.grid(True, alpha=0.2)
            ax.legend(loc='upper left', fontsize=7, ncol=2)
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

            # Salvar para Tabela
            stats_resumo.append({
                'Ponto': nome, 
                'Vel_InSAR (mm/ano)': slope_i*365.25, 
                'Vel_Geo (mm/ano)': slope_g*365.25, 
                'RMSE (mm)': rmse,
                'R2': r_val_i**2
            })

    plt.tight_layout()
    plt.show()

# ==============================================================================
# 6. TABELA RESUMO FINAL
# ==============================================================================
print("\n--- TABELA RESUMO FINAL (PERÍODO 2019-2023) ---")
df_resumo = pd.DataFrame(stats_resumo)
print(df_resumo.to_string(index=False))

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import fiona
import contextily as ctx
from scipy.spatial import cKDTree
from scipy import stats
from sklearn.metrics import mean_squared_error

# ==============================================================================
# 1. CONFIGURAÇÕES E CAMINHOS (DOIS PERÍODOS)
# ==============================================================================
# --- Período 2019-2023 ---
PATH_ASC_19_23 = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC_19_23 = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# --- Período 2018-2022 ---
PATH_ASC_18_22 = "data/alqueva_calibrated_asc_desc_2018_2022/EGMS_L2b_147_0224_IW2_VV_2018_2022_1/EGMS_L2b_147_0224_IW2_VV_2018_2022_1.csv"
PATH_DESC_18_22 = "data/alqueva_calibrated_asc_desc_2018_2022/EGMS_L2b_052_0848_IW2_VV_2018_2022_1/EGMS_L2b_052_0848_IW2_VV_2018_2022_1.csv"

PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_NIVEL = 'data/nivelamento.xlsx'

# Parâmetros Espaciais
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
RAIO_IDW, RAIO_AMOSTRA = 15, 15
IDW_K, IDW_P = 3, 2

# ==============================================================================
# 2. FUNÇÕES DE PROCESSAMENTO
# ==============================================================================
def melt_robust(df, val_name):
    meta = ['easting', 'northing', 'latitude', 'longitude', 'incidence_angle', 'track_angle', 'pid', 'p_id']
    present_meta = [c for c in meta if c in df.columns]
    date_cols = [c for c in df.columns if c not in present_meta]
    m = df.melt(id_vars=present_meta, value_vars=date_cols, var_name='date', value_name=val_name).dropna()
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    m[val_name] = pd.to_numeric(m[val_name], errors='coerce')
    return m.dropna(subset=['date', val_name])

def load_filter(path):
    df = pd.read_csv(path)
    df['easting'] = pd.to_numeric(df['easting'], errors='coerce')
    df['northing'] = pd.to_numeric(df['northing'], errors='coerce')
    return df[(df['northing'] >= norte_min-100) & (df['northing'] <= norte_max+100) & 
              (df['easting'] >= este_min-100) & (df['easting'] <= este_max+100)].dropna(subset=['easting', 'northing'])

def interp_ps(df, dates):
    dfs = []
    t_x = dates.view(np.int64) 
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        xp, fp = g['date'].values.view(np.int64), g['disp'].values.astype(np.float64)
        res = pd.DataFrame({'easting': x, 'northing': y, 'date': dates, 'disp': np.interp(t_x, xp, fp),
                           'inc': g['incidence_angle'].iloc[0], 'lon': g['longitude'].iloc[0], 'lat': g['latitude'].iloc[0],
                           'pid': g['pid'].iloc[0] if 'pid' in g.columns else 0})
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r, k, p):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(src[['easting', 'northing']].values)
        dist, idx = tree.query(tgt[['easting', 'northing']].values, k=k, distance_upper_bound=r)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1 / (d_i[mask]**p)
            vals.append(np.sum(w * src.iloc[i_i[mask]]['disp']) / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[mask]]['inc']) / np.sum(w))
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna()

# ==============================================================================
# 3. CARREGAMENTO E UNIÃO DOS PERÍODOS (2018-2023)
# ==============================================================================
print("1/5 - Integrando períodos 2018-2022 e 2019-2023...")

# Processar Órbita Ascendente
asc_18_22 = melt_robust(load_filter(PATH_ASC_18_22), 'disp')
asc_19_23 = melt_robust(load_filter(PATH_ASC_19_23), 'disp')
asc_l = pd.concat([asc_18_22, asc_19_23]).drop_duplicates(subset=['pid', 'date']).sort_values(['pid', 'date'])

# Processar Órbita Descendente
desc_18_22 = melt_robust(load_filter(PATH_DESC_18_22), 'disp')
desc_19_23 = melt_robust(load_filter(PATH_DESC_19_23), 'disp')
desc_l = pd.concat([desc_18_22, desc_19_23]).drop_duplicates(subset=['pid', 'date']).sort_values(['pid', 'date'])

# Datas comuns para interpolação (cobrem todo o intervalo 2018-2023)
common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')

print(f"Série temporal unificada: {asc_l['date'].min().date()} a {asc_l['date'].max().date()}")

# ==============================================================================
# 4. PROCESSAMENTO InSAR (INTERPOLAÇÃO E IDW)
# ==============================================================================
print("2/5 - Processando Interpolação e Decomposição Vertical...")
asc_i = interp_ps(asc_l, common_dates)
desc_i = interp_ps(desc_l, common_dates)

manual_dv_df = idw_calc(desc_i, asc_i, r=RAIO_IDW, k=IDW_K, p=IDW_P)

# Cálculo do Deslocamento Vertical ($d_V$)
manual_dv_df['dV_final'] = (manual_dv_df['disp_desc']*np.sin(np.deg2rad(manual_dv_df['inc'])) + 
                            manual_dv_df['disp']*np.sin(np.deg2rad(manual_dv_df['theta_desc']))) / \
                            np.sin(np.deg2rad(manual_dv_df['inc']) + np.deg2rad(manual_dv_df['theta_desc']))

gdf_manual = gpd.GeoDataFrame(manual_dv_df, geometry=gpd.points_from_xy(manual_dv_df.lon, manual_dv_df.lat), crs="EPSG:4326").to_crs(epsg=3763)

# ==============================================================================
# 5. CARREGAMENTO GEODESIA E KML
# ==============================================================================
print("3/5 - Carregando Geodesia e buffers KML...")
df_geo_raw = pd.read_excel(PATH_NIVEL)
df_geo_raw['data'] = pd.to_datetime(df_geo_raw['data'])
df_geo_raw['valor_corrigido'] = pd.to_numeric(df_geo_raw['deslocamento (m)'].astype(str).str.replace(',', '.'), errors='coerce')

fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml = gpd.read_file(PATH_KML, driver='KML')
gdf_pts = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_circs = gdf_pts.copy()
gdf_circs.geometry = gdf_pts.geometry.buffer(RAIO_AMOSTRA)

# Amostragem Espacial
final_series = {}
for nome in gdf_circs['Name'].unique():
    m_data = gpd.sjoin(gdf_manual, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
    if not m_data.empty:
        final_series[nome] = m_data.groupby('date')['dV_final'].mean()

# ==============================================================================
# 6. ANÁLISE HISTÓRICA E GRÁFICOS (SINC. 2019)
# ==============================================================================

print("4/5 - Gerando Gráficos de Comparação Histórica...")
inst_comuns = [n for n in final_series.keys() if n in df_geo_raw['instrumento'].unique()]
stats_resumo = []

if inst_comuns:
    fig, axes = plt.subplots(len(inst_comuns), 1, figsize=(14, 4 * len(inst_comuns)))
    if len(inst_comuns) == 1: axes = [axes]
    
    for i, nome in enumerate(inst_comuns):
        ax = axes[i]
        
        # --- InSAR Unificado ---
        s_insar = final_series[nome].sort_index()
        # Sincronização em 2019 (conforme pedido)
        s_insar_2019 = s_insar[s_insar.index.year >= 2019]
        if not s_insar_2019.empty:
            ref_insar = s_insar_2019.iloc[0]
            insar_sync = s_insar - ref_insar
        else:
            insar_sync = s_insar - s_insar.iloc[0]
        
        # --- Geodesia ---
        s_geo_full = df_geo_raw[df_geo_raw['instrumento'] == nome].sort_values('data')
        s_geo_2019 = s_geo_full[s_geo_full['data'].dt.year >= 2019]
        
        if not s_geo_2019.empty and not s_insar.empty:
            ref_geo_2019 = s_geo_2019['valor_corrigido'].iloc[0]
            geo_hist_sync = s_geo_full['valor_corrigido'] - ref_geo_2019
            
            # Cálculo de Tendências (2019-2023) para a tabela
            days_insar = (s_insar_2019.index - s_insar_2019.index[0]).days
            slope_i, intercept_i, r_val_i, _, _ = stats.linregress(days_insar, (s_insar_2019 - s_insar_2019.iloc[0]).values)
            
            days_geo_2019 = (s_geo_2019['data'] - s_geo_2019['data'].iloc[0]).dt.days
            vals_geo_2019 = s_geo_2019['valor_corrigido'] - ref_geo_2019
            slope_g, intercept_g, r_val_g, _, _ = stats.linregress(days_geo_2019, vals_geo_2019)

            # RMSE
            insar_interp = np.interp(days_geo_2019, (s_insar_2019.index - s_insar_2019.index[0]).days, (s_insar_2019 - s_insar_2019.iloc[0]).values)
            rmse = np.sqrt(mean_squared_error(vals_geo_2019, insar_interp))

            # PLOTAGEM
            ax.plot(s_geo_full['data'][s_geo_full['data'].dt.year < 2019], geo_hist_sync[s_geo_full['data'].dt.year < 2019], color='gray', ls='--', alpha=0.4, label='Histórico Geodesia')
            ax.scatter(s_geo_2019['data'], geo_hist_sync[s_geo_full['data'].dt.year >= 2019], color='red', marker='D', s=35, label='Geodesia (2019-2023)', zorder=5)
            
            # InSAR Unificado (2018-2023)
            ax.plot(insar_sync.index, insar_sync.values, color='tab:blue', lw=1.2, label='InSAR Unificado (2018-2023)')
            
            # Linhas de Tendência (2019+)
            ax.plot(s_insar_2019.index, (intercept_i + slope_i * days_insar), color='darkblue', lw=2, label=f'Trend InSAR ({slope_i*365.25:.2f} mm/a)')

            ax.axhline(0, color='black', lw=1); ax.axvline(pd.Timestamp('2019-01-01'), color='green', lw=1.5, ls='--')
            ax.set_title(f"Ponto {nome}: Série Integrada (Sync 2019)", fontweight='bold')
            ax.set_ylabel("Variação (mm)"); ax.grid(True, alpha=0.2); ax.legend(loc='upper left', fontsize=7, ncol=2)

            stats_resumo.append({'Ponto': nome, 'Vel_InSAR (mm/ano)': slope_i*365.25, 'Vel_Geo (mm/ano)': slope_g*365.25, 'RMSE (mm)': rmse, 'R2': r_val_i**2})

    plt.tight_layout()
    plt.show()

# ==============================================================================
# 7. TABELA RESUMO
# ==============================================================================

print("\n--- TABELA RESUMO FINAL (UNIÃO 2018-2023) ---")
df_resumo = pd.DataFrame(stats_resumo)
print(df_resumo.to_string(index=False))

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import fiona
import contextily as ctx
from scipy.spatial import cKDTree
from scipy import stats
from sklearn.metrics import mean_squared_error

# ==============================================================================
# 1. CONFIGURAÇÕES E CAMINHOS (UNIÃO 2018-2023)
# ==============================================================================
PATH_ASC_19_23 = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC_19_23 = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
PATH_ASC_18_22 = "data/alqueva_calibrated_asc_desc_2018_2022/EGMS_L2b_147_0224_IW2_VV_2018_2022_1/EGMS_L2b_147_0224_IW2_VV_2018_2022_1.csv"
PATH_DESC_18_22 = "data/alqueva_calibrated_asc_desc_2018_2022/EGMS_L2b_052_0848_IW2_VV_2018_2022_1/EGMS_L2b_052_0848_IW2_VV_2018_2022_1.csv"
PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_NIVEL = 'data/nivelamento.xlsx'

# Parâmetros Espaciais
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
RAIO_IDW, RAIO_AMOSTRA = 50, 25
IDW_K, IDW_P = 10, 2

# ==============================================================================
# 2. FUNÇÕES DE PROCESSAMENTO
# ==============================================================================
def melt_robust(df, val_name):
    meta = ['easting', 'northing', 'latitude', 'longitude', 'incidence_angle', 'track_angle', 'pid', 'p_id']
    present_meta = [c for c in meta if c in df.columns]
    date_cols = [c for c in df.columns if c not in present_meta]
    m = df.melt(id_vars=present_meta, value_vars=date_cols, var_name='date', value_name=val_name).dropna()
    m['date'] = pd.to_datetime(m['date'], format='%Y%m%d', errors='coerce')
    m[val_name] = pd.to_numeric(m[val_name], errors='coerce')
    return m.dropna(subset=['date', val_name])

def load_filter(path):
    df = pd.read_csv(path)
    df['easting'] = pd.to_numeric(df['easting'], errors='coerce')
    df['northing'] = pd.to_numeric(df['northing'], errors='coerce')
    return df[(df['northing'] >= norte_min-100) & (df['northing'] <= norte_max+100) & 
              (df['easting'] >= este_min-100) & (df['easting'] <= este_max+100)].dropna(subset=['easting', 'northing'])

def interp_ps(df, dates):
    dfs = []
    t_x = dates.view(np.int64) 
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        xp, fp = g['date'].values.view(np.int64), g['disp'].values.astype(np.float64)
        res = pd.DataFrame({'easting': x, 'northing': y, 'date': dates, 'disp': np.interp(t_x, xp, fp),
                           'inc': g['incidence_angle'].iloc[0], 'lon': g['longitude'].iloc[0], 'lat': g['latitude'].iloc[0],
                           'pid': g['pid'].iloc[0] if 'pid' in g.columns else 0})
        dfs.append(res)
    return pd.concat(dfs) if dfs else pd.DataFrame()

def idw_calc(source, target, r, k, p):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(src[['easting', 'northing']].values)
        dist, idx = tree.query(tgt[['easting', 'northing']].values, k=k, distance_upper_bound=r)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            mask = np.isfinite(d_i)
            if not np.any(mask): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1 / (d_i[mask]**p)
            vals.append(np.sum(w * src.iloc[i_i[mask]]['disp']) / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[mask]]['inc']) / np.sum(w))
        tgt['disp_desc'], tgt['theta_desc'] = vals, thetas
        out.append(tgt)
    return pd.concat(out).dropna()

# ==============================================================================
# 3. UNIÃO E PROCESSAMENTO InSAR (2018-2023)
# ==============================================================================
print("1/5 - Integrando períodos 2018-2023...")
asc_l = pd.concat([melt_robust(load_filter(PATH_ASC_18_22), 'disp'), 
                   melt_robust(load_filter(PATH_ASC_19_23), 'disp')]).drop_duplicates(subset=['pid', 'date'])
desc_l = pd.concat([melt_robust(load_filter(PATH_DESC_18_22), 'disp'), 
                    melt_robust(load_filter(PATH_DESC_19_23), 'disp')]).drop_duplicates(subset=['pid', 'date'])

common_dates = pd.date_range(start=asc_l['date'].min(), end=asc_l['date'].max(), freq='MS')
asc_i, desc_i = interp_ps(asc_l, common_dates), interp_ps(desc_l, common_dates)

manual_dv_df = idw_calc(desc_i, asc_i, r=RAIO_IDW, k=IDW_K, p=IDW_P)
manual_dv_df['dV_final'] = (manual_dv_df['disp_desc']*np.sin(np.deg2rad(manual_dv_df['inc'])) + 
                            manual_dv_df['disp']*np.sin(np.deg2rad(manual_dv_df['theta_desc']))) / \
                            np.sin(np.deg2rad(manual_dv_df['inc']) + np.deg2rad(manual_dv_df['theta_desc']))

gdf_manual = gpd.GeoDataFrame(manual_dv_df, geometry=gpd.points_from_xy(manual_dv_df.lon, manual_dv_df.lat), crs="EPSG:4326").to_crs(epsg=3763)

# ==============================================================================
# 4. CARREGAMENTO GEODESIA E KML
# ==============================================================================
print("2/5 - Carregando Geodesia...")
df_geo_raw = pd.read_excel(PATH_NIVEL)
df_geo_raw['data'] = pd.to_datetime(df_geo_raw['data'])
df_geo_raw['valor_corrigido'] = pd.to_numeric(df_geo_raw['deslocamento (m)'].astype(str).str.replace(',', '.'), errors='coerce')

fiona.drvsupport.supported_drivers['KML'] = 'rw'
gdf_kml = gpd.read_file(PATH_KML, driver='KML')
gdf_pts = gdf_kml[gdf_kml.geometry.type == 'Point'].copy().to_crs(epsg=3763)
gdf_circs = gdf_pts.copy(); gdf_circs.geometry = gdf_pts.geometry.buffer(RAIO_AMOSTRA)

final_series = {}
for nome in gdf_circs['Name'].unique():
    m_data = gpd.sjoin(gdf_manual, gdf_circs[gdf_circs['Name']==nome], how="inner", predicate="within")
    if not m_data.empty:
        final_series[nome] = m_data.groupby('date')['dV_final'].mean()

# ==============================================================================
# 5. GRÁFICOS COM ESCALA UNIFORME E ZERO EM 2018
# ==============================================================================
print("3/5 - Calculando limites globais e tendências...")
inst_comuns = [n for n in final_series.keys() if n in df_geo_raw['instrumento'].unique()]
stats_resumo = []
plot_data = {}
all_values = []

# Primeiro Passo: Sincronização e cálculo de limites
for nome in inst_comuns:
    s_insar = final_series[nome].sort_index()
    # Sincronização em 2018 (primeiro registo disponível)
    ref_insar = s_insar.iloc[0]
    insar_sync = s_insar - ref_insar
    
    s_geo_full = df_geo_raw[df_geo_raw['instrumento'] == nome].sort_values('data')
    s_geo_2018 = s_geo_full[s_geo_full['data'].dt.year >= 2018]
    
    if not s_geo_2018.empty:
        ref_geo_2018 = s_geo_2018['valor_corrigido'].iloc[0]
        geo_sync = s_geo_full['valor_corrigido'] - ref_geo_2018
        
        # Guardar para limites
        all_values.extend(insar_sync.values)
        all_values.extend(geo_sync[s_geo_full['data'].dt.year >= 2004].values) # Considerar histórico
        
        plot_data[nome] = {
            'insar': insar_sync, 'geo_full': s_geo_full, 'geo_sync': geo_sync,
            'geo_ref_date': s_geo_2018['data'].iloc[0], 'insar_ref_date': s_insar.index[0]
        }

# Escala Uniforme
y_min, y_max = min(all_values) - 2, max(all_values) + 2

# Segundo Passo: Plotagem
fig, axes = plt.subplots(len(plot_data), 1, figsize=(14, 4.5 * len(plot_data)))
if len(plot_data) == 1: axes = [axes]

for i, (nome, data) in enumerate(plot_data.items()):
    ax = axes[i]
    insar = data['insar']
    geo_full = data['geo_full']
    geo_sync = data['geo_sync']
    
    # 1. Geodesia completa (Vermelho)
    ax.scatter(geo_full['data'], geo_sync, color='red', marker='D', s=35, label='Geodesia (Histórico + Atual)', zorder=5)
    
    # 2. InSAR (Azul)
    ax.plot(insar.index, insar.values, color='tab:blue', lw=1.5, label='InSAR Integrado (2018-2023)')
    
    # 3. Linha Verde no primeiro registo InSAR
    ax.axvline(data['insar_ref_date'], color='green', lw=2, ls='--', label=f'Início InSAR ({data["insar_ref_date"].date()})')
    
    # 4. Cálculo e Plotagem de Tendências (Período InSAR 2018-2023)
    days_i = (insar.index - insar.index[0]).days
    slope_i, intercept_i, r_val_i, _, _ = stats.linregress(days_i, insar.values)
    ax.plot(insar.index, (intercept_i + slope_i * days_i), color='darkblue', lw=2.5, label=f'Trend InSAR ({slope_i*365.25:.2f} mm/a)')
    
    geo_2018 = geo_full[geo_full['data'] >= data['insar_ref_date']]
    if len(geo_2018) > 1:
        days_g = (geo_2018['data'] - geo_2018['data'].iloc[0]).dt.days
        slope_g, intercept_g, _, _, _ = stats.linregress(days_g, (geo_2018['valor_corrigido'] - geo_2018['valor_corrigido'].iloc[0]).values)
        ax.plot(geo_2018['data'], (intercept_g + slope_g * days_g), color='darkred', ls=':', lw=2.5, label=f'Trend Geo ({slope_g*365.25:.2f} mm/a)')
        
        # RMSE
        insar_interp = np.interp(days_g, days_i, insar.values)
        rmse = np.sqrt(mean_squared_error((geo_2018['valor_corrigido'] - geo_2018['valor_corrigido'].iloc[0]).values, insar_interp))
    else:
        slope_g, rmse = 0, np.nan

    # Estética
    ax.set_ylim(y_min, y_max) # <--- APLICAÇÃO DA ESCALA UNIFORME
    ax.axhline(0, color='black', lw=1)
    ax.set_title(f"Ponto {nome}: Comparação de Tendências (Zero em 2018)", fontweight='bold')
    ax.set_ylabel("Variação (mm)"); ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=8, ncol=2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    
    stats_resumo.append({'Ponto': nome, 'Vel_InSAR (mm/ano)': slope_i*365.25, 'Vel_Geo (mm/ano)': slope_g*365.25, 'RMSE (mm)': rmse, 'R2_InSAR': r_val_i**2})

plt.tight_layout(); plt.show()

# ==============================================================================
# 6. TABELA RESUMO
# ==============================================================================
print("\n--- TABELA RESUMO FINAL (ZERO EM 2018) ---")
df_resumo = pd.DataFrame(stats_resumo)
print(df_resumo.to_string(index=False))